In [4]:
import json
from pathlib import Path
import pandas as pd

RAW_DIR = Path("../data/raw")
all_dfs = []

for filepath in sorted(RAW_DIR.glob("*.json")):
    with open(filepath) as f:
        data = json.load(f)
    df = pd.DataFrame(data["hourly"])
    df["city_file"] = filepath.stem
    df["api_latitude"] = data["latitude"]
    df["api_longitude"] = data["longitude"]
    all_dfs.append(df)

combined = pd.concat(all_dfs, ignore_index=True)
combined["time"] = pd.to_datetime(combined["time"])
print(f"Total rows: {len(combined):,}")
combined.head()

Total rows: 1,680


,time,temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m,wind_direction_10m,pressure_msl,weather_code,city_file,api_latitude,api_longitude
0,2026-05-04 00:00:00,17.5,48,0.0,14.9,278,1015.5,2,alexandria_20260504T135020Z,31.1875,29.9375
1,2026-05-04 01:00:00,17.5,51,0.0,18.0,286,1015.6,2,alexandria_20260504T135020Z,31.1875,29.9375
2,2026-05-04 02:00:00,16.8,60,0.0,19.6,284,1015.6,2,alexandria_20260504T135020Z,31.1875,29.9375
3,2026-05-04 03:00:00,16.5,66,0.0,20.6,282,1015.7,2,alexandria_20260504T135020Z,31.1875,29.9375
4,2026-05-04 04:00:00,15.7,75,0.0,23.4,284,1016.0,3,alexandria_20260504T135020Z,31.1875,29.9375


In [2]:
nulls = combined.isnull().sum()
nulls_pct = (nulls / len(combined) * 100).round(2)
quality = pd.DataFrame({"null_count": nulls, "null_pct": nulls_pct})
quality[quality["null_count"] > 0]

,null_count,null_pct


In [3]:
numeric_cols = combined.select_dtypes(include="number").columns
combined[numeric_cols].describe().T[["min", "max", "mean"]]

,min,max,mean
temperature_2m,-1.500000,35.000000,16.935655
relative_humidity_2m,18.000000,100.000000,66.438690
precipitation,0.000000,6.000000,0.059690
wind_speed_10m,0.300000,38.500000,12.109286
wind_direction_10m,1.000000,360.000000,183.857738
pressure_msl,993.300000,1028.100000,1014.431905
weather_code,0.000000,95.000000,9.313095
api_latitude,-33.989456,64.139565,18.104423
api_longitude,-73.993080,151.195510,30.058192


In [4]:
# Compare what you requested vs. what the API returned
requested = {
    "cairo":      (30.0444,  31.2357),
    "alexandria": (31.2001,  29.9187),
    "london":     (51.5074,  -0.1278),
    # ...
}

for city_file in combined["city_file"].unique():
    city_key = city_file.split("_")[0]
    if city_key in requested:
        req_lat, req_lon = requested[city_key]
        got = combined[combined["city_file"] == city_file].iloc[0]
        delta_lat = abs(req_lat - got["api_latitude"])
        delta_lon = abs(req_lon - got["api_longitude"])
        print(f"{city_key:12} requested ({req_lat:.4f}, {req_lon:.4f}) "
              f"got ({got['api_latitude']:.4f}, {got['api_longitude']:.4f}) "
              f"delta=({delta_lat:.4f}, {delta_lon:.4f})")

alexandria   requested (31.2001, 29.9187) got (31.1875, 29.9375) delta=(0.0126, 0.0188)
cairo        requested (30.0444, 31.2357) got (30.0625, 31.2500) delta=(0.0181, 0.0143)
london       requested (51.5074, -0.1278) got (51.5115, -0.1308) delta=(0.0041, 0.0030)


In [5]:
combined.groupby("city_file")["time"].agg(["min", "max", "count"])

,min,max,count
city_file,,,
alexandria_20260503T004508Z,2026-05-03,2026-05-09 23:00:00,168
cairo_20260503T004508Z,2026-05-03,2026-05-09 23:00:00,168
cape_town_20260503T004508Z,2026-05-03,2026-05-09 23:00:00,168
london_20260503T004508Z,2026-05-03,2026-05-09 23:00:00,168
mumbai_20260503T004508Z,2026-05-03,2026-05-09 23:00:00,168
new_york_20260503T004508Z,2026-05-03,2026-05-09 23:00:00,168
reykjavik_20260503T004508Z,2026-05-03,2026-05-09 23:00:00,168
sao_paulo_20260503T004508Z,2026-05-03,2026-05-09 23:00:00,168
sydney_20260503T004508Z,2026-05-03,2026-05-09 23:00:00,168


In [ ]:
# 1. null counts
print("Missing values:")
print(combined.isnull().sum())

# 2. فحص الإحصائيات (درجات الحرارة، الرطوبة، إلخ)
combined.describe()

Missing values:
time                    0
temperature_2m          0
relative_humidity_2m    0
precipitation           0
wind_speed_10m          0
wind_direction_10m      0
pressure_msl            0
weather_code            0
city_file               0
api_latitude            0
api_longitude           0
dtype: int64


,time,temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m,wind_direction_10m,pressure_msl,weather_code,api_latitude,api_longitude
count,1680,1680.000000,1680.000000,1680.000000,1680.000000,1680.000000,1680.000000,1680.000000,1680.000000,1680.000000
mean,2026-05-07 11:30:00,17.175833,65.267857,0.055595,11.871786,179.926786,1015.214048,12.360119,18.104423,30.058192
min,2026-05-04 00:00:00,-3.100000,15.000000,0.000000,0.000000,0.000000,994.900000,0.000000,-33.989456,-73.993080
25%,2026-05-05 17:45:00,12.600000,52.000000,0.000000,6.900000,56.000000,1011.700000,0.000000,-23.514938,-21.971603
50%,2026-05-07 11:30:00,16.800000,66.000000,0.000000,10.800000,189.500000,1016.300000,1.000000,30.625000,24.182234
75%,2026-05-09 05:15:00,21.500000,80.000000,0.000000,15.500000,295.000000,1018.700000,3.000000,40.710335,72.852910
max,2026-05-10 23:00:00,36.500000,100.000000,7.200000,44.000000,360.000000,1030.600000,95.000000,64.139565,151.195510
std,NaN,7.725063,18.343189,0.294750,6.709249,118.739077,5.822667,26.095202,33.945498,69.999743


In [6]:
# (Value Ranges)
cols_to_check = ['temperature_2m', 'relative_humidity_2m', 'wind_speed_10m']

print("--- Statistical Summary for Value Ranges ---")
print(combined[cols_to_check].describe())

invalid_humidity = combined[(combined['relative_humidity_2m'] < 0) | (combined['relative_humidity_2m'] > 100)]
print(f"\nNumber of invalid humidity records: {len(invalid_humidity)}")

--- Statistical Summary for Value Ranges ---
       temperature_2m  relative_humidity_2m  wind_speed_10m
count     1680.000000           1680.000000     1680.000000
mean        17.175833             65.267857       11.871786
std          7.725063             18.343189        6.709249
min         -3.100000             15.000000        0.000000
25%         12.600000             52.000000        6.900000
50%         16.800000             66.000000       10.800000
75%         21.500000             80.000000       15.500000
max         36.500000            100.000000       44.000000

Number of invalid humidity records: 0


In [5]:
# 3. (Coordinate check)
print("--- Unique Coordinates per City ---")
coords_check = combined.groupby('city_file')[['api_latitude', 'api_longitude']].nunique()
print(coords_check)

--- Unique Coordinates per City ---
                             api_latitude  api_longitude
city_file                                               
alexandria_20260504T135020Z             1              1
cairo_20260504T135020Z                  1              1
cape_town_20260504T135020Z              1              1
london_20260504T135020Z                 1              1
mumbai_20260504T135020Z                 1              1
new_york_20260504T135020Z               1              1
reykjavik_20260504T135020Z              1              1
sao_paulo_20260504T135020Z              1              1
sydney_20260504T135020Z                 1              1
tokyo_20260504T135020Z                  1              1


In [5]:
# 4. (Time Coverage)
print("--- Time Coverage per City ---")

time_summary = combined.groupby('city_file').agg(
    start_time=('time', 'min'),
    end_time=('time', 'max'),
    total_hours=('time', 'count')
)

print(time_summary)

missing_hours = time_summary[time_summary['total_hours'] != 168]
if missing_hours.empty:
    print("\n✅ Perfect! All cities have full 168-hour coverage.")
else:
    print("\n⚠️ Warning: Some cities have missing hours!")
    print(missing_hours)

--- Time Coverage per City ---
                            start_time            end_time  total_hours
city_file                                                              
alexandria_20260504T135020Z 2026-05-04 2026-05-10 23:00:00          168
cairo_20260504T135020Z      2026-05-04 2026-05-10 23:00:00          168
cape_town_20260504T135020Z  2026-05-04 2026-05-10 23:00:00          168
london_20260504T135020Z     2026-05-04 2026-05-10 23:00:00          168
mumbai_20260504T135020Z     2026-05-04 2026-05-10 23:00:00          168
new_york_20260504T135020Z   2026-05-04 2026-05-10 23:00:00          168
reykjavik_20260504T135020Z  2026-05-04 2026-05-10 23:00:00          168
sao_paulo_20260504T135020Z  2026-05-04 2026-05-10 23:00:00          168
sydney_20260504T135020Z     2026-05-04 2026-05-10 23:00:00          168
tokyo_20260504T135020Z      2026-05-04 2026-05-10 23:00:00          168

✅ Perfect! All cities have full 168-hour coverage.
